In [7]:
import os
import numpy as np
import pandas as pd
import json
import itertools
import joblib

In [8]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [9]:
# --------------------------------------------------
# 1. LOAD DATA
# --------------------------------------------------

if os.path.exists("/home/patel-krish/anaconda3/BrainyBeam/heart.csv"):
    df = pd.read_csv("/home/patel-krish/anaconda3/BrainyBeam/heart.csv")
elif os.path.exists("/home/patel-krish/anaconda3/BrainyBeam/heart.csv"):
    df = pd.read_csv("/home/patel-krish/anaconda3/BrainyBeam/heart.csv")
else:
    raise FileNotFoundError(
        "Dataset not found. Place heart(2).csv or heart.csv "
        "in the current notebook folder."
    )

df.columns = df.columns.str.strip()
df = df.drop_duplicates()
df = df.dropna(subset=["target"])

X = df.drop(columns=["target"]).apply(
    pd.to_numeric, errors="coerce"
)
X = X.replace([np.inf, -np.inf], np.nan)

y = pd.to_numeric(df["target"], errors="coerce")
valid = y.notna()

X = X.loc[valid].reset_index(drop=True)
y = y.loc[valid].astype(int).reset_index(drop=True)

if y.nunique() != 2:
    raise ValueError("This code requires a binary target.")

if y.value_counts().min() < 5:
    raise ValueError(
        "Each target class needs at least five samples for 5-fold CV."
    )

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Dataset loaded:", df.shape)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Dataset loaded: (302, 14)
Training rows: 241
Testing rows: 61


In [10]:
# --------------------------------------------------
# 2. DEFINE MODELS
# --------------------------------------------------

models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=2000,
            random_state=42
        ))
    ]),

    "Decision Tree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier(random_state=42))
    ]),

    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(random_state=42))
    ]),

    "K-Nearest Neighbors": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier())
    ]),

    "Support Vector Machine": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", SVC())
    ])
}

In [11]:
# --------------------------------------------------
# 3. DEFINE PARAMETER GRIDS
# --------------------------------------------------

parameter_grids = {
    "Logistic Regression": {
        "model__C": [0.1, 1, 10],
        "model__solver": ["liblinear", "lbfgs"],
        "model__class_weight": [None, "balanced"]
    },

    "Decision Tree": {
        "model__max_depth": [3, 5, 10, None],
        "model__criterion": ["gini", "entropy"],
        "model__min_samples_split": [2, 5, 10]
    },

    "Random Forest": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [5, 10, None],
        "model__min_samples_split": [2, 5],
        "model__max_features": ["sqrt", None]
    },

    "K-Nearest Neighbors": {
        "model__n_neighbors": [3, 5, 7, 9],
        "model__weights": ["uniform", "distance"],
        "model__p": [1, 2]
    },

    "Support Vector Machine": {
        "model__C": [0.1, 1, 10],
        "model__kernel": ["linear", "rbf"],
        "model__gamma": ["scale", "auto"]
    }
}

In [12]:

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

all_search_results = []
best_overall_f1 = -1
best_model_name = None
best_params = None
best_model = None

for model_name, base_model in models.items():

    print("\nTuning:", model_name)

    grid = parameter_grids[model_name]
    keys = list(grid.keys())

    combinations = [
        dict(zip(keys, values))
        for values in itertools.product(
            *[grid[key] for key in keys]
        )
    ]

    best_model_f1 = -1
    best_model_params = None

    for number, params in enumerate(combinations, start=1):

        fold_scores = []

        for train_idx, valid_idx in cv.split(X_train, y_train):

            X_fold_train = X_train.iloc[train_idx]
            X_fold_valid = X_train.iloc[valid_idx]
            y_fold_train = y_train.iloc[train_idx]
            y_fold_valid = y_train.iloc[valid_idx]

            current_model = clone(base_model)
            current_model.set_params(**params)
            current_model.fit(X_fold_train, y_fold_train)

            predictions = current_model.predict(X_fold_valid)

            fold_scores.append(
                f1_score(
                    y_fold_valid,
                    predictions,
                    zero_division=0
                )
            )

        mean_f1 = float(np.mean(fold_scores))

        all_search_results.append({
            "Model": model_name,
            "Parameters": str(params),
            "Mean_CV_F1": mean_f1
        })

        if mean_f1 > best_model_f1:
            best_model_f1 = mean_f1
            best_model_params = params.copy()

        print(
            f"\rCombination {number}/{len(combinations)} "
            f"- Mean F1: {mean_f1:.4f}",
            end=""
        )

    print()
    print("Best parameters:", best_model_params)
    print("Best CV F1:", round(best_model_f1, 4))

    if best_model_f1 > best_overall_f1:
        best_overall_f1 = best_model_f1
        best_model_name = model_name
        best_params = best_model_params.copy()
        best_model = clone(base_model)
        best_model.set_params(**best_params)


Tuning: Logistic Regression
Combination 12/12 - Mean F1: 0.8709
Best parameters: {'model__C': 10, 'model__solver': 'liblinear', 'model__class_weight': 'balanced'}
Best CV F1: 0.8709

Tuning: Decision Tree
Combination 24/24 - Mean F1: 0.8093
Best parameters: {'model__max_depth': 3, 'model__criterion': 'entropy', 'model__min_samples_split': 2}
Best CV F1: 0.8367

Tuning: Random Forest
Combination 24/24 - Mean F1: 0.8489
Best parameters: {'model__n_estimators': 100, 'model__max_depth': 5, 'model__min_samples_split': 2, 'model__max_features': 'sqrt'}
Best CV F1: 0.8752

Tuning: K-Nearest Neighbors
Combination 16/16 - Mean F1: 0.8584
Best parameters: {'model__n_neighbors': 7, 'model__weights': 'distance', 'model__p': 1}
Best CV F1: 0.8674

Tuning: Support Vector Machine
Combination 12/12 - Mean F1: 0.8180
Best parameters: {'model__C': 1, 'model__kernel': 'rbf', 'model__gamma': 'scale'}
Best CV F1: 0.8691


In [13]:
search_results = pd.DataFrame(all_search_results)
search_results = search_results.sort_values(
    "Mean_CV_F1", ascending=False
).reset_index(drop=True)

search_results.to_csv(
    "manual_grid_search_results.csv",
    index=False
)

In [14]:
best_model.fit(X_train, y_train)

print("\nBEST MODEL:", best_model_name)
print("BEST PARAMETERS:", best_params)
print("BEST CROSS-VALIDATION F1:", round(best_overall_f1, 4))


BEST MODEL: Random Forest
BEST PARAMETERS: {'model__n_estimators': 100, 'model__max_depth': 5, 'model__min_samples_split': 2, 'model__max_features': 'sqrt'}
BEST CROSS-VALIDATION F1: 0.8752


In [15]:
test_predictions = best_model.predict(X_test)

test_metrics = {
    "Accuracy": float(accuracy_score(y_test, test_predictions)),
    "Precision": float(precision_score(
        y_test, test_predictions, zero_division=0
    )),
    "Recall": float(recall_score(
        y_test, test_predictions, zero_division=0
    )),
    "F1": float(f1_score(
        y_test, test_predictions, zero_division=0
    ))
}

print("\nFINAL TEST METRICS")
for metric, value in test_metrics.items():
    print(f"{metric}: {value:.4f}")


FINAL TEST METRICS
Accuracy: 0.7705
Precision: 0.7879
Recall: 0.7879
F1: 0.7879


In [16]:
MODEL_VERSION = "1.1.0"

safe_name = best_model_name.lower().replace(" ", "_")
export_dir = os.path.join(
    "model_exports", safe_name, "v" + MODEL_VERSION
)
os.makedirs(export_dir, exist_ok=True)

model_path = os.path.join(
    export_dir, safe_name + "_v" + MODEL_VERSION + ".joblib"
)

joblib.dump(
    best_model,
    model_path,
    compress=("gzip", 3),
    protocol=5
)

def sha256_file(path):
    import hashlib
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(8192), b""):
            digest.update(chunk)

    return digest.hexdigest()

metadata = {
    "model_name": best_model_name,
    "version": MODEL_VERSION,
    "best_parameters": best_params,
    "mean_cv_f1": best_overall_f1,
    "test_metrics": test_metrics,
    "features": list(X.columns),
    "target": "target",
    "model_file": os.path.basename(model_path),
    "sha256": sha256_file(model_path)
}

with open(
    os.path.join(export_dir, "manifest.json"),
    "w",
    encoding="utf-8"
) as file:
    json.dump(metadata, file, indent=4)

# Verify that the saved model can be loaded
loaded_model = joblib.load(model_path)
assert np.array_equal(
    loaded_model.predict(X_test),
    test_predictions
)

print("\nTUNING AND EXPORT COMPLETED")
print("Model path:", os.path.abspath(model_path))
print("Results CSV:", os.path.abspath("manual_grid_search_results.csv"))
print("Manifest:", os.path.abspath(
    os.path.join(export_dir, "manifest.json")
))
print("Model reload verification: passed")



TUNING AND EXPORT COMPLETED
Model path: /home/patel-krish/anaconda3/BrainyBeam/Heart_Diesease/model_exports/random_forest/v1.1.0/random_forest_v1.1.0.joblib
Results CSV: /home/patel-krish/anaconda3/BrainyBeam/Heart_Diesease/manual_grid_search_results.csv
Manifest: /home/patel-krish/anaconda3/BrainyBeam/Heart_Diesease/model_exports/random_forest/v1.1.0/manifest.json
Model reload verification: passed
